In [1]:
import os
import sys
import glob
import warnings
import time
import numpy as np
from scipy import stats, signal
from scipy.io import loadmat
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import logging
import joblib

warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
class Config:
    DATASET_ROOT = './CZU-MHAD'

    SKELETON_DIR = "skeleton_mat"
    INERTIAL_DIR = "sensor_mat"
    DEPTH_DIR = "depth_mat"

    SUBFOLDERS = ["depth_mat", "sensor_mat", "skeleton_mat"]
    FEATURES_DIR = "features_noise_1.0_30"

    NUM_ACTIONS = 22
    NUM_SUBJECTS = 5
    NUM_TRIALS = 5
    NUM_JOINTS = 25

    ALL_SUBJECTS = ['cx', 'myj', 'zyh', 'cyy', 'qyh']
    TRAIN_SUBJECTS = ['cx', 'myj', 'zyh']
    TEST_SUBJECTS = ['cyy', 'qyh']

    # Inertial feature extraction parameters (inspired by VISTA paper)
    INERTIAL_WINDOW_SIZE = 50
    INERTIAL_WINDOW_OVERLAP = 0.5

    DEPTH_SAMPLE_FRAMES = 10

    # Random Forest parameters
    RF_N_ESTIMATORS = 300
    RF_MAX_DEPTH = None
    RF_MIN_SAMPLES_SPLIT = 2
    RF_MIN_SAMPLES_LEAF = 1
    RF_RANDOM_STATE = 42
    RF_N_JOBS = 1

    USE_SKELETON = True
    USE_INERTIAL = True
    USE_DEPTH = True

    LOSS_RATE = 0.30
    LOSS_SEED = 42


In [3]:
def extract_numeric_array(mat_value):
    if isinstance(mat_value, np.ndarray):
        if mat_value.dtype == object:
            if mat_value.size == 1:
                inner = mat_value.flat[0]
                if isinstance(inner, np.ndarray):
                    return inner.astype(np.float64)
                else:
                    return np.array(inner, dtype=np.float64)

            parts = []
            for item in mat_value.flat:
                if isinstance(item, np.ndarray) and item.size > 0:
                    parts.append(item.astype(np.float64))

            if parts:
                try:
                    return np.concatenate(parts, axis=0)
                except ValueError:
                    return parts[0]
            return None

        if np.issubdtype(mat_value.dtype, np.number):
            return mat_value.astype(np.float64)

        try:
            return mat_value.astype(np.float64)
        except (ValueError, TypeError):
            return None

    return None


class CZUMHADLoader:
    """
    File naming: {subject_id}_a{action_id}_t{trial_id}.mat
    Subjects: 'cx', 'myj', 'zyh', 'cyy', 'qyh'
    """

    def __init__(self, config):
        self.config = config
        self.root = config.DATASET_ROOT

    def load_all_data(self):
        sensor_dir = os.path.join(self.root, self.config.INERTIAL_DIR)
        all_files = [f for f in os.listdir(sensor_dir) if f.endswith(".mat")]
        subjects_found = sorted(list({f.split("_")[0] for f in all_files}))
        logger.info(f"Subjects found: {subjects_found}")
        logger.info(f"Total .mat files in {self.config.INERTIAL_DIR}: {len(all_files)}")

        # Find samples that exist in all 3 modality folders
        valid_samples = []
        skipped = 0
        for file in sorted(all_files):
            base_name = file.replace(".mat", "")
            exists_in_all = all(
                os.path.exists(os.path.join(self.root, sub, base_name + ".mat"))
                for sub in self.config.SUBFOLDERS
            )
            if not exists_in_all:
                skipped += 1
                if skipped <= 5:
                    logger.warning(f"Skipping incomplete sample: {base_name}")
                continue

            parts = base_name.split("_")
            subj = parts[0]
            action = int(parts[1][1:])
            trial = int(parts[2][1:])
            valid_samples.append({
                'subject': subj, 'action': action, 'trial': trial,
                'basename': base_name,
            })

        if skipped > 0:
            logger.warning(f"Skipped {skipped} incomplete samples total")
        logger.info(f"Samples with all 3 modalities: {len(valid_samples)}")

        if valid_samples:
            bn0 = valid_samples[0]['basename']
            for sub in self.config.SUBFOLDERS:
                fpath = os.path.join(self.root, sub, bn0 + ".mat")
                try:
                    mat = loadmat(fpath)
                    keys = [k for k in mat.keys() if not k.startswith("__")]
                    for k in keys:
                        v = mat[k]
                        logger.info(f"  DEBUG {sub}/{bn0}.mat key='{k}': "
                                    f"type={type(v).__name__}, "
                                    f"dtype={v.dtype if hasattr(v, 'dtype') else 'N/A'}, "
                                    f"shape={v.shape if hasattr(v, 'shape') else 'N/A'}")
                        if hasattr(v, 'dtype') and v.dtype == object and v.size <= 4:
                            for idx in range(min(v.size, 2)):
                                inner = v.flat[idx]
                                logger.info(f"    [flat {idx}]: type={type(inner).__name__}, "
                                            f"dtype={inner.dtype if hasattr(inner, 'dtype') else 'N/A'}, "
                                            f"shape={inner.shape if hasattr(inner, 'shape') else 'N/A'}")
                except Exception as e:
                    logger.info(f"  DEBUG error inspecting {sub}/{bn0}.mat: {e}")

        # Load data
        samples = {}
        load_errors = 0
        first_errors = {}
        for sample_meta in valid_samples:
            bn = sample_meta['basename']
            try:
                # Load skeleton
                skel_path = os.path.join(self.root, "skeleton_mat", bn + ".mat")
                mat = loadmat(skel_path)
                key = [k for k in mat.keys() if not k.startswith("__")][0]
                skel_raw = extract_numeric_array(mat[key])
                if skel_raw is None:
                    raise ValueError(f"Cannot extract numeric array from skeleton key '{key}'")
                
                if skel_raw.ndim == 2 and skel_raw.shape[1] == 100:
                    col_idx = [j*4 + c for j in range(25) for c in range(3)]
                    skel_raw = skel_raw[:, col_idx]  # → (frames, 75)

                # Load sensor
                sensor_path = os.path.join(self.root, "sensor_mat", bn + ".mat")
                mat = loadmat(sensor_path)
                key = [k for k in mat.keys() if not k.startswith("__")][0]
                sensor_raw = extract_numeric_array(mat[key])
                if sensor_raw is None:
                    raise ValueError(f"Cannot extract numeric array from sensor key '{key}'")

                # Depth: filepath only
                depth_path = os.path.join(self.root, "depth_mat", bn + ".mat")

                samples[bn] = {
                    'action': sample_meta['action'],
                    'subject': sample_meta['subject'],
                    'trial': sample_meta['trial'],
                    'skeleton': skel_raw,
                    'inertial': sensor_raw,
                    'depth': depth_path,
                }
            except Exception as e:
                load_errors += 1
                err_msg = str(e)
                err_type = err_msg[:60]
                if err_type not in first_errors:
                    first_errors[err_type] = bn
                if load_errors <= 3:
                    logger.warning(f"Error loading {bn}: {e}")

        if load_errors > 3:
            logger.warning(f"Total loading errors: {load_errors}")
            logger.warning(f"Error types seen:")
            for err_type, first_bn in first_errors.items():
                logger.warning(f"  '{err_type}...' (first seen in: {first_bn})")

        logger.info(f"Successfully loaded {len(samples)} complete samples")

        if samples:
            first = next(iter(samples.values()))
            logger.info(f"  First sample skeleton: shape={first['skeleton'].shape}, dtype={first['skeleton'].dtype}")
            logger.info(f"  First sample sensor:   shape={first['inertial'].shape}, dtype={first['inertial'].dtype}")
            logger.info(f"  First sample depth:    filepath (lazy)")

        return samples


In [4]:
# =============================================================================
# SECTION 2b: SIMULATED SENSOR NOISE – per-sample helpers
# =============================================================================
#
# Each modality has its own helper called inside the pipeline loop.
#
# Noise model (identical to UTD-MHAD noise version):
#   noise std = noise_strength x std(selected unit's raw values)
#   Noise is ADDED to the original values (not replacing them).
#   Units with zero variance fall back to std = 1e-3.
#   noise_strength = 0.2 (set as NOISE_STRENGTH in MultimodalHARPipeline)
#
# CZU-MHAD shape notes (same fixes as the loss version):
#
#   Skeleton  : stored as 2D (frames, 75) — 25 joints x 3 coords flat.
#               apply_skeleton_noise() handles both 2D and 3D:
#               - 2D (frames, 75): noises columns in groups of 3 (per joint)
#               - 3D (frames, J, 3): noises joint slices directly
#
#   Inertial  : stored as (10, 1) — only 1 real channel.
#               For single-channel data, noise is applied to TIME STEPS
#               (rows) instead of channels so each sample gets different
#               corruption, matching the fix from the loss version.
#
#   Depth     : shape (frames, H, W) — unchanged, was already correct.
# =============================================================================

import math
import gc


def _n_units_to_noise(n_units: int, rate: float) -> int:
    return max(1, math.ceil(n_units * rate))


def _add_gaussian_noise(data: np.ndarray, rng: np.random.RandomState,
                        noise_strength: float = 1.0) -> np.ndarray:
    """
    Add Gaussian noise to `data` IN-PLACE.

    Noise std = noise_strength x std(data).
    Falls back to std = 1e-3 for zero-variance units.

    Args:
        data           : numpy array of any shape (modified in-place)
        rng            : numpy RandomState for reproducibility
        noise_strength : fraction of data std to use as noise std
    Returns:
        The same array with noise added.
    """
    sigma = noise_strength * np.std(data)
    if sigma < 1e-8:
        sigma = 1e-3
    data += rng.normal(loc=0.0, scale=sigma, size=data.shape)
    return data


def apply_skeleton_noise(skel: np.ndarray, rate: float,
                         noise_strength: float,
                         rng: np.random.RandomState) -> np.ndarray:
    """
    Add Gaussian noise to random joints in a skeleton sequence IN-PLACE.

    Handles both CZU-MHAD 2D format (frames, J*3) and standard 3D
    format (frames, J, 3).

    Noise is added to ALL frames of each selected joint.
    Noise std = noise_strength x std(joint data over all frames & coords).

    Args:
        skel           : (frames, J*3) or (frames, J, 3)
        rate           : fraction of joints to corrupt  (e.g. 0.50)
        noise_strength : noise std as a fraction of joint std  (e.g. 0.2)
        rng            : caller-owned RandomState (advances each call)
    Returns:
        same array (for chaining)
    """
    if skel is None or skel.size == 0:
        return skel

    if skel.ndim == 2:
        # CZU format: (frames, J*3)  e.g. (frames, 75) for 25 joints
        n_frames, n_cols = skel.shape
        if n_cols % 3 != 0:
            return skel   # unrecognised layout — leave untouched
        n_joints = n_cols // 3
        n_noisy  = _n_units_to_noise(n_joints, rate)
        joints   = rng.choice(n_joints, size=n_noisy, replace=False)
        for j in joints:
            _add_gaussian_noise(skel[:, j*3 : j*3+3], rng, noise_strength)
        logger.debug(f"Skeleton noise (2D): noised joints {joints.tolist()} ({n_noisy}/{n_joints})")

    elif skel.ndim == 3:
        if skel.shape[1] == 0:
            return skel
        n_joints = skel.shape[1]
        n_noisy  = _n_units_to_noise(n_joints, rate)
        joints   = rng.choice(n_joints, size=n_noisy, replace=False)
        for j in joints:
            _add_gaussian_noise(skel[:, j, :], rng, noise_strength)
        logger.debug(f"Skeleton noise (3D): noised joints {joints.tolist()} ({n_noisy}/{n_joints})")

    return skel


def apply_inertial_noise(iner: np.ndarray, rate: float,
                          noise_strength: float,
                          rng: np.random.RandomState) -> np.ndarray:
    """
    Add Gaussian noise to random channels (or time steps for single-channel
    data) in an inertial sequence IN-PLACE.

    CZU-MHAD inertial data is (10, 1) — one channel.  For single-channel
    data, noise is applied to TIME STEPS so each sample gets different
    corruption rather than always noising the same single channel identically.

    Args:
        iner           : (time_steps, n_channels)
        rate           : fraction of channels (or time steps) to corrupt
        noise_strength : noise std as a fraction of unit std
        rng            : caller-owned RandomState
    """
    if iner is None or iner.ndim != 2:
        return iner

    n_rows, n_cols = iner.shape

    if n_cols <= 1:
        # Single-channel: apply noise across time steps (rows)
        if n_rows == 0:
            return iner
        n_noisy = _n_units_to_noise(n_rows, rate)
        rows    = rng.choice(n_rows, size=n_noisy, replace=False)
        for row in rows:
            _add_gaussian_noise(iner[row:row+1, :], rng, noise_strength)
        logger.debug(f"Inertial noise (time-step): noised rows {rows.tolist()} ({n_noisy}/{n_rows})")
    else:
        # Multi-channel: apply noise across channels
        n_noisy = _n_units_to_noise(n_cols, rate)
        chs     = rng.choice(n_cols, size=n_noisy, replace=False)
        for ch in chs:
            _add_gaussian_noise(iner[:, ch], rng, noise_strength)
        logger.debug(f"Inertial noise (channel): noised channels {chs.tolist()} ({n_noisy}/{n_cols})")

    return iner


def apply_depth_noise(depth: np.ndarray, rate: float,
                      noise_strength: float,
                      rng: np.random.RandomState) -> np.ndarray:
    """
    Add Gaussian noise to random frames in a depth volume IN-PLACE.

    Handles both (H, W, frames) and (frames, H, W) axis orderings.
    Noise std = noise_strength x std(selected frame pixel values).

    Args:
        depth          : 3-D depth array
        rate           : fraction of frames to corrupt
        noise_strength : noise std as a fraction of frame std
        rng            : caller-owned RandomState
    """
    if depth is None or depth.ndim != 3:
        return depth
    sh = depth.shape
    if sh[2] <= sh[0] and sh[2] <= sh[1]:     # (H, W, frames)
        n_frames = sh[2]
        n_noisy  = _n_units_to_noise(n_frames, rate)
        frs      = rng.choice(n_frames, size=n_noisy, replace=False)
        for fr in frs:
            _add_gaussian_noise(depth[:, :, fr], rng, noise_strength)
    else:                                       # (frames, H, W)
        n_frames = sh[0]
        n_noisy  = _n_units_to_noise(n_frames, rate)
        frs      = rng.choice(n_frames, size=n_noisy, replace=False)
        for fr in frs:
            _add_gaussian_noise(depth[fr, :, :], rng, noise_strength)
    logger.debug(f"Depth noise: noised {n_noisy}/{n_frames} frames")
    return depth


In [5]:
# =============================================================================
# SECTION 3: SKELETON FEATURE EXTRACTION
# =============================================================================
#
# Two dataset-specific extractors that produce IDENTICAL feature dimensions
# and semantics.  Only the raw joint-index constants differ because UTD-MHAD
# (Kinect v1, 20 joints) and CZU-MHAD (Kinect v2, 25 joints) number their
# joints differently.
#
# Joint correspondence (the 20 shared body parts):
#
#   Body part            UTD-MHAD   CZU-MHAD
#   ─────────────────    ────────   ────────
#   head                    0          3
#   shoulder center/neck    1          2
#   spine / spine mid       2          1
#   hip center/spine base   3          0
#   left shoulder           4          4
#   left elbow              5          5
#   left wrist              6          6
#   left hand               7          7
#   right shoulder          8          8
#   right elbow             9          9
#   right wrist            10         10
#   right hand             11         11
#   left hip               12         12
#   left knee              13         13
#   left ankle             14         14
#   left foot              15         15
#   right hip              16         16
#   right knee             17         17
#   right ankle            18         18
#   right foot             19         19
#
# CZU-MHAD additionally has joints 20-24 (spine shoulder, hand tips, thumbs)
# which are NOT used in feature extraction, so both extractors output the
# exact same feature vector length.
# =============================================================================


class _SkeletonFeatureExtractorBase:
    """
    Base class containing all shared feature-extraction logic.

    Subclasses only need to set the joint-index class attributes for their
    specific dataset.  All feature computation references these semantic
    names, so the resulting feature vectors have identical meaning and
    dimensionality across datasets.

    Preprocessing pipeline:
        1. Detect missing joints (all-zero, NaN, Inf)
        2. Temporal linear interpolation to fill gaps
        3. Light Gaussian smoothing (sigma=1) to reduce sensor jitter

    Feature categories:
        1. Normalized joint position statistics  (VISTA-inspired)
        2. Pairwise joint distance statistics
        3. Joint angle statistics
        4. Bone (segment) length statistics
        5. Velocity statistics
        6. Acceleration statistics
        7. Covariance matrix features           (MSDFE-inspired)
        8. Centre-of-mass trajectory statistics
        9. Global motion energy
    """

    HEAD = None
    SHOULDER_CENTER = None   # neck in CZU-MHAD
    SPINE = None             # spine_mid in CZU-MHAD
    HIP_CENTER = None        # spine_base in CZU-MHAD
    SHOULDER_LEFT = None
    ELBOW_LEFT = None
    WRIST_LEFT = None
    HAND_LEFT = None
    SHOULDER_RIGHT = None
    ELBOW_RIGHT = None
    WRIST_RIGHT = None
    HAND_RIGHT = None
    HIP_LEFT = None
    KNEE_LEFT = None
    ANKLE_LEFT = None
    FOOT_LEFT = None
    HIP_RIGHT = None
    KNEE_RIGHT = None
    ANKLE_RIGHT = None
    FOOT_RIGHT = None

    NUM_JOINTS = None        # 20 for UTD, 20 used (out of 25) for CZU
    RAW_JOINTS = None        # total joints in the raw data (20 or 25)
    RAW_COLS = None          # expected flat columns (60 or 75)

    def __init__(self):
        # ---- Built from semantic names (resolved at subclass import) ----
        self.JOINT_PAIRS = [
            (self.HAND_LEFT,     self.HAND_RIGHT),      # left hand – right hand
            (self.FOOT_LEFT,     self.FOOT_RIGHT),      # left foot – right foot
            (self.HAND_LEFT,     self.HIP_CENTER),      # left hand – hip
            (self.HAND_RIGHT,    self.HIP_CENTER),      # right hand – hip
            (self.HAND_LEFT,     self.HEAD),             # left hand – head
            (self.HAND_RIGHT,    self.HEAD),             # right hand – head
            (self.FOOT_LEFT,     self.HIP_CENTER),      # left foot – hip
            (self.FOOT_RIGHT,    self.HIP_CENTER),      # right foot – hip
            (self.WRIST_LEFT,    self.WRIST_RIGHT),     # left wrist – right wrist
            (self.ELBOW_LEFT,    self.ELBOW_RIGHT),     # left elbow – right elbow
            (self.SHOULDER_LEFT, self.SHOULDER_RIGHT),  # left shoulder – right shoulder
            (self.KNEE_LEFT,     self.KNEE_RIGHT),      # left knee – right knee
            (self.HIP_LEFT,      self.HIP_RIGHT),       # left hip – right hip
            (self.HAND_LEFT,     self.FOOT_LEFT),       # left hand – left foot
            (self.HAND_RIGHT,    self.FOOT_RIGHT),      # right hand – right foot
            (self.HEAD,          self.HIP_CENTER),      # head – hip (body height proxy)
        ]

        self.ANGLE_TRIPLETS = [
            (self.SHOULDER_LEFT,  self.ELBOW_LEFT,     self.WRIST_LEFT),    # L shoulder-elbow-wrist
            (self.SHOULDER_RIGHT, self.ELBOW_RIGHT,    self.WRIST_RIGHT),   # R shoulder-elbow-wrist
            (self.HIP_LEFT,       self.KNEE_LEFT,      self.ANKLE_LEFT),    # L hip-knee-ankle
            (self.HIP_RIGHT,      self.KNEE_RIGHT,     self.ANKLE_RIGHT),   # R hip-knee-ankle
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER, self.SHOULDER_RIGHT),# L shoulder – center – R shoulder
            (self.ELBOW_LEFT,     self.SHOULDER_LEFT,   self.SHOULDER_CENTER),
            (self.ELBOW_RIGHT,    self.SHOULDER_RIGHT,  self.SHOULDER_CENTER),
            (self.HIP_CENTER,     self.SPINE,           self.SHOULDER_CENTER),# hip – spine – shoulder center
            (self.HIP_LEFT,       self.HIP_CENTER,      self.HIP_RIGHT),     # L hip – center – R hip
            (self.SHOULDER_LEFT,  self.SHOULDER_CENTER,  self.HEAD),          # L shoulder – center – head
            (self.SHOULDER_RIGHT, self.SHOULDER_CENTER,  self.HEAD),          # R shoulder – center – head
        ]

        self.BONE_PAIRS = [
            (self.HIP_CENTER,     self.SPINE),
            (self.SPINE,          self.SHOULDER_CENTER),
            (self.SHOULDER_CENTER,self.HEAD),
            (self.SHOULDER_CENTER,self.SHOULDER_LEFT),
            (self.SHOULDER_LEFT,  self.ELBOW_LEFT),
            (self.ELBOW_LEFT,     self.WRIST_LEFT),
            (self.WRIST_LEFT,     self.HAND_LEFT),
            (self.SHOULDER_CENTER,self.SHOULDER_RIGHT),
            (self.SHOULDER_RIGHT, self.ELBOW_RIGHT),
            (self.ELBOW_RIGHT,    self.WRIST_RIGHT),
            (self.WRIST_RIGHT,    self.HAND_RIGHT),
            (self.HIP_CENTER,     self.HIP_LEFT),
            (self.HIP_LEFT,       self.KNEE_LEFT),
            (self.KNEE_LEFT,      self.ANKLE_LEFT),
            (self.ANKLE_LEFT,     self.FOOT_LEFT),
            (self.HIP_CENTER,     self.HIP_RIGHT),
            (self.HIP_RIGHT,      self.KNEE_RIGHT),
            (self.KNEE_RIGHT,     self.ANKLE_RIGHT),
            (self.ANKLE_RIGHT,    self.FOOT_RIGHT),
        ]

        # Key joints for position / velocity / acceleration stats
        self.KEY_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.SHOULDER_LEFT, self.ELBOW_LEFT, self.WRIST_LEFT, self.HAND_LEFT,
            self.SHOULDER_RIGHT, self.ELBOW_RIGHT, self.WRIST_RIGHT, self.HAND_RIGHT,
            self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

        # Key joints for covariance matrix
        self.COV_JOINTS = [
            self.HIP_CENTER, self.SPINE, self.SHOULDER_CENTER, self.HEAD,
            self.HAND_LEFT, self.HAND_RIGHT, self.FOOT_LEFT, self.FOOT_RIGHT,
        ]

    # =====================================================================
    # Preprocessing
    # =====================================================================

    def _detect_missing(self, skel):
        is_nan = np.any(np.isnan(skel), axis=2)
        is_inf = np.any(np.isinf(skel), axis=2)
        is_zero = np.all(np.abs(skel) < 1e-10, axis=2)
        return is_nan | is_inf | is_zero


    def _interpolate_missing(self, skel, missing_mask):
        skel = skel.copy()
        n_frames, n_joints, _ = skel.shape

        for j in range(n_joints):
            if not np.any(missing_mask[:, j]):
                continue

            valid_idx = np.where(~missing_mask[:, j])[0]
            if len(valid_idx) == 0:
                continue  # handled below

            for axis in range(3):
                skel[:, j, axis] = np.interp(
                    np.arange(n_frames), valid_idx, skel[valid_idx, j, axis]
                )

        # Joints missing in ALL frames: copy from kinematic parent
        all_missing = np.where(np.all(missing_mask, axis=0))[0]
        if len(all_missing) > 0:
            parent_map = {}
            for p, c in self.BONE_PAIRS:
                parent_map[c] = p
            for j in all_missing:
                parent = parent_map.get(j, None)
                if parent is not None and not np.all(missing_mask[:, parent]):
                    skel[:, j, :] = skel[:, parent, :]
                else:
                    skel[:, j, :] = 0.0

        return skel


    def _preprocess(self, skel):
        missing_mask = self._detect_missing(skel)
        skel = np.nan_to_num(skel, nan=0.0, posinf=0.0, neginf=0.0)

        n_missing = np.sum(missing_mask)
        if n_missing > 0:
            logger.debug(
                f"Skeleton preprocessing: {n_missing} missing joint-frames "
                f"({100 * n_missing / missing_mask.size:.1f}%) — interpolating"
            )
            skel = self._interpolate_missing(skel, missing_mask)

        # Light temporal Gaussian smoothing to reduce sensor jitter
        if skel.shape[0] >= 5:
            from scipy.ndimage import gaussian_filter1d
            skel = gaussian_filter1d(skel, sigma=1.0, axis=0)

        return skel


    def _normalize_skeleton(self, skel):
        """
        Translate to hip center, scale by torso length (hip → shoulder center).

        Args:
            skel: (frames, J, 3)
        Returns:
            normalized skeleton (frames, J, 3)
        """
        hip = skel[:, self.HIP_CENTER:self.HIP_CENTER + 1, :]
        skel_norm = skel - hip

        torso_vec = (skel_norm[:, self.SHOULDER_CENTER, :]
                     - skel_norm[:, self.HIP_CENTER, :])
        torso_len = np.linalg.norm(torso_vec, axis=1, keepdims=True)
        torso_len = np.clip(torso_len, 1e-6, None)
        skel_norm = skel_norm / torso_len[:, np.newaxis, :]

        return skel_norm


    def _compute_joint_distances(self, skel):
        """Pairwise joint distances over time. → (frames, 16)"""
        distances = []
        for j1, j2 in self.JOINT_PAIRS:
            distances.append(np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1))
        return np.array(distances).T


    def _compute_joint_angles(self, skel):
        angles = []
        for j1, j2, j3 in self.ANGLE_TRIPLETS:
            v1 = skel[:, j1, :] - skel[:, j2, :]
            v2 = skel[:, j3, :] - skel[:, j2, :]
            cos_a = np.sum(v1 * v2, axis=1) / (
                np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8
            )
            angles.append(np.arccos(np.clip(cos_a, -1.0, 1.0)))
        return np.array(angles).T


    def _compute_bone_lengths(self, skel):
        lengths = []
        for j1, j2 in self.BONE_PAIRS:
            lengths.append(np.linalg.norm(skel[:, j1, :] - skel[:, j2, :], axis=1))
        return np.array(lengths).T


    def _compute_velocity(self, skel):
        if skel.shape[0] < 2:
            return np.zeros_like(skel)
        vel = np.diff(skel, axis=0)
        return np.vstack([vel, vel[-1:]])


    def _compute_acceleration(self, skel):
        vel = self._compute_velocity(skel)
        if vel.shape[0] < 2:
            return np.zeros_like(vel)
        acc = np.diff(vel, axis=0)
        return np.vstack([acc, acc[-1:]])


    def _temporal_statistics(self, signal_2d):
        # 9 statistics per column: mean, std, RMS, skew, kurtosis, range, median, Q1, Q3
        features = []
        for col in range(signal_2d.shape[1]):
            s = signal_2d[:, col]
            features.extend([
                np.mean(s),
                np.std(s),
                np.sqrt(np.mean(s ** 2)),
                stats.skew(s) if len(s) > 2 else 0.0,
                stats.kurtosis(s) if len(s) > 3 else 0.0,
                np.max(s) - np.min(s),
                np.median(s),
                np.percentile(s, 25),
                np.percentile(s, 75),
            ])
        return np.array(features)


    def _covariance_features(self, skel):
        # Upper-triangle of covariance matrix over 8 key joints (24 coords)
        flat = skel.reshape(skel.shape[0], -1)
        idx = []
        for j in self.COV_JOINTS:
            idx.extend([j * 3, j * 3 + 1, j * 3 + 2])
        flat_key = flat[:, idx]

        if flat_key.shape[0] < 2:
            cov = np.zeros((flat_key.shape[1], flat_key.shape[1]))
        else:
            cov = np.cov(flat_key.T)

        return cov[np.triu_indices(cov.shape[0])]


    def _to_frames_joints_3(self, skel):
        if skel.ndim == 2:
            n_frames, n_cols = skel.shape
            if n_cols >= self.RAW_COLS:
                return skel[:, :self.RAW_COLS].reshape(n_frames, self.RAW_JOINTS, 3)
            return None

        if skel.ndim != 3:
            return None

        s = skel.shape
        J = self.RAW_JOINTS

        if s[1] == J and s[2] == 3:
            return skel
        if s[0] == J and s[2] == 3:
            return np.transpose(skel, (1, 0, 2))
        if s[2] == J and s[1] == 3:
            return np.transpose(skel, (0, 2, 1))
        if s[0] == J and s[1] == 3:
            return np.transpose(skel, (2, 0, 1))
        if s[0] == 3 and s[1] == J:
            return np.transpose(skel, (2, 1, 0))
        if s[2] == J:
            return np.transpose(skel, (2, 0, 1))

        return np.transpose(skel, (2, 0, 1))


    def extract(self, skel):
        if skel is None or skel.size == 0:
            return None

        skel = self._to_frames_joints_3(skel)
        if skel is None or skel.shape[0] == 0:
            return None

        if skel.shape[1] > 20:
            skel = skel[:, :20, :]

        skel = self._preprocess(skel)

        skel_norm = self._normalize_skeleton(skel)

        # 1. Position statistics on key joints
        flat_pos = skel_norm.reshape(skel_norm.shape[0], -1)
        key_idx = []
        for j in self.KEY_JOINTS:
            key_idx.extend([j * 3, j * 3 + 1, j * 3 + 2])
        pos_feats = self._temporal_statistics(flat_pos[:, key_idx])

        # 2. Joint distances
        dist_feats = self._temporal_statistics(self._compute_joint_distances(skel_norm))

        # 3. Joint angles
        angle_feats = self._temporal_statistics(self._compute_joint_angles(skel_norm))

        # 4. Bone lengths
        bone_feats = self._temporal_statistics(self._compute_bone_lengths(skel_norm))

        # 5. Velocity
        vel = self._compute_velocity(skel_norm).reshape(skel_norm.shape[0], -1)
        vel_feats = self._temporal_statistics(vel[:, key_idx])

        # 6. Acceleration
        acc = self._compute_acceleration(skel_norm).reshape(skel_norm.shape[0], -1)
        acc_feats = self._temporal_statistics(acc[:, key_idx])

        # 7. Covariance
        cov_feats = self._covariance_features(skel_norm)

        # 8. Centre of mass
        com = np.mean(skel_norm, axis=1)
        com_feats = self._temporal_statistics(com)

        # 9. Motion energy
        if skel_norm.shape[0] > 1:
            total_disp = np.sum(
                np.linalg.norm(np.diff(skel_norm, axis=0), axis=2), axis=1
            )
        else:
            total_disp = np.array([0.0])
        motion_energy = np.array([
            np.mean(total_disp), np.std(total_disp),
            np.max(total_disp), np.sum(total_disp),
        ])

        all_feats = np.concatenate([
            pos_feats, dist_feats, angle_feats, bone_feats,
            vel_feats, acc_feats, cov_feats, com_feats, motion_energy,
        ])

        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)


# =========================================================================
# UTD-MHAD  (Kinect v1, 20 joints)
# =========================================================================

class SkeletonFeatureExtractor(_SkeletonFeatureExtractorBase):
    """
    Skeleton feature extractor for the UTD-MHAD dataset.

    UTD-MHAD joint indices (Kinect v1, 20 joints):
        0: head, 1: shoulder center, 2: spine, 3: hip center,
        4: left shoulder, 5: left elbow, 6: left wrist, 7: left hand,
        8: right shoulder, 9: right elbow, 10: right wrist, 11: right hand,
        12: left hip, 13: left knee, 14: left ankle, 15: left foot,
        16: right hip, 17: right knee, 18: right ankle, 19: right foot
    """
    HEAD             = 0
    SHOULDER_CENTER  = 1
    SPINE            = 2
    HIP_CENTER       = 3
    SHOULDER_LEFT    = 4
    ELBOW_LEFT       = 5
    WRIST_LEFT       = 6
    HAND_LEFT        = 7
    SHOULDER_RIGHT   = 8
    ELBOW_RIGHT      = 9
    WRIST_RIGHT      = 10
    HAND_RIGHT       = 11
    HIP_LEFT         = 12
    KNEE_LEFT        = 13
    ANKLE_LEFT       = 14
    FOOT_LEFT        = 15
    HIP_RIGHT        = 16
    KNEE_RIGHT       = 17
    ANKLE_RIGHT      = 18
    FOOT_RIGHT       = 19

    NUM_JOINTS = 20
    RAW_JOINTS = 20
    RAW_COLS   = 60   # 20 × 3


# =========================================================================
# CZU-MHAD  (Kinect v2, 25 joints — only first 20 used for features)
# =========================================================================

class CZUSkeletonFeatureExtractor(_SkeletonFeatureExtractorBase):
    """
    Skeleton feature extractor for the CZU-MHAD dataset.

    CZU-MHAD joint indices (Kinect v2, 25 joints):
        0: spine base, 1: spine mid, 2: neck, 3: head,
        4: shoulder left, 5: elbow left, 6: wrist left, 7: hand left,
        8: shoulder right, 9: elbow right, 10: wrist right, 11: hand right,
        12: hip left, 13: knee left, 14: ankle left, 15: foot left,
        16: hip right, 17: knee right, 18: ankle right, 19: foot right,
        20: spine shoulder, 21: hand tip left, 22: thumb left,
        23: hand tip right, 24: thumb right

    Only the first 20 joints are used for feature extraction so the output
    has identical dimension and semantics to SkeletonFeatureExtractor.
    """
    HEAD             = 3
    SHOULDER_CENTER  = 2   # neck
    SPINE            = 1   # spine mid
    HIP_CENTER       = 0   # spine base
    SHOULDER_LEFT    = 4
    ELBOW_LEFT       = 5
    WRIST_LEFT       = 6
    HAND_LEFT        = 7
    SHOULDER_RIGHT   = 8
    ELBOW_RIGHT      = 9
    WRIST_RIGHT      = 10
    HAND_RIGHT       = 11
    HIP_LEFT         = 12
    KNEE_LEFT        = 13
    ANKLE_LEFT       = 14
    FOOT_LEFT        = 15
    HIP_RIGHT        = 16
    KNEE_RIGHT       = 17
    ANKLE_RIGHT      = 18
    FOOT_RIGHT       = 19

    NUM_JOINTS = 20    # features computed on 20 joints
    RAW_JOINTS = 25    # raw data has 25 joints
    RAW_COLS   = 75    # 25 × 3


In [6]:
# =============================================================================
# SECTION 4: INERTIAL FEATURE EXTRACTION
# =============================================================================

class InertialFeatureExtractor:
    """
    Extracts features from inertial sensor data (accelerometer + gyroscope).

    Directly inspired by the VISTA paper (Fiorini et al., 2022):
    - Time-domain features: mean, std, RMS, skewness, kurtosis, SMA, power
    - Additional: zero-crossing rate, peak count, signal energy
    - Frequency-domain: dominant frequency, spectral entropy, band energy

    The VISTA paper demonstrated these features on wrist + finger IMUs
    and showed 73-81% accuracy with individual sensors and higher with fusion.
    """

    def __init__(self, config):
        self.window_size = config.INERTIAL_WINDOW_SIZE
        self.overlap = config.INERTIAL_WINDOW_OVERLAP

    def _signal_magnitude_area(self, data):
        """
        Signal Magnitude Area (SMA) - used in VISTA paper.
        SMA = (1/N) * sum(|ax| + |ay| + |az|)
        """
        return np.mean(np.sum(np.abs(data), axis=1))

    def _signal_power(self, s):
        """Signal power = mean of squared values."""
        return np.mean(s**2)

    def _zero_crossing_rate(self, s):
        """Count zero crossings normalized by length."""
        s_centered = s - np.mean(s)
        return np.sum(np.abs(np.diff(np.sign(s_centered)))) / (2.0 * len(s))

    def _peak_count(self, s):
        """Count number of peaks in signal."""
        peaks, _ = signal.find_peaks(s)
        return len(peaks) / len(s)

    def _spectral_entropy(self, s, fs=50):
        """Compute spectral entropy of the signal."""
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        psd_norm = psd / (np.sum(psd) + 1e-12)
        psd_norm = psd_norm[psd_norm > 0]
        return -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

    def _dominant_frequency(self, s, fs=50):
        """Find the dominant frequency component."""
        if len(s) < 4:
            return 0.0
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        return freqs[np.argmax(psd)]

    def _frequency_band_energy(self, s, fs=50, bands=[(0, 5), (5, 15), (15, 25)]):
        """Energy in different frequency bands."""
        if len(s) < 4:
            return [0.0] * len(bands)
        freqs, psd = signal.welch(s, fs=fs, nperseg=min(len(s), 256))
        energies = []
        for low, high in bands:
            mask = (freqs >= low) & (freqs < high)
            energies.append(np.sum(psd[mask]))
        return energies

    def _extract_channel_features(self, s):
        """
        Extract comprehensive features from a single channel.
        Based on VISTA paper's feature set + extensions.
        """
        features = []

        # Time-domain (VISTA paper features)
        features.append(np.mean(s))                          # Mean
        features.append(np.std(s))                           # Standard deviation
        features.append(np.sqrt(np.mean(s**2)))              # RMS
        features.append(stats.skew(s) if len(s) > 2 else 0) # Skewness
        features.append(stats.kurtosis(s) if len(s) > 3 else 0)  # Kurtosis
        features.append(self._signal_power(s))               # Power
        features.append(np.max(s) - np.min(s))               # Range
        features.append(np.median(s))                        # Median
        features.append(np.mean(np.abs(s)))                  # Mean absolute value
        features.append(np.percentile(s, 25))                # Q1
        features.append(np.percentile(s, 75))                # Q3
        features.append(np.percentile(s, 75) - np.percentile(s, 25))  # IQR
        features.append(self._zero_crossing_rate(s))         # Zero-crossing rate
        features.append(self._peak_count(s))                 # Peak count

        # Frequency-domain
        features.append(self._dominant_frequency(s))         # Dominant freq
        features.append(self._spectral_entropy(s))           # Spectral entropy
        features.extend(self._frequency_band_energy(s))      # Band energies

        return features

    def _extract_window_features(self, window):
        """
        Extract features from a single window of inertial data.

        Args:
            window: (window_size, 6) array [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        Returns:
            1D feature vector
        """
        features = []

        acc = window[:, :3]   # Accelerometer
        gyro = window[:, 3:]  # Gyroscope

        # Per-channel features for all 6 channels
        for ch in range(6):
            features.extend(self._extract_channel_features(window[:, ch]))

        # Accelerometer magnitude
        acc_mag = np.linalg.norm(acc, axis=1)
        features.extend(self._extract_channel_features(acc_mag))

        # Gyroscope magnitude
        gyro_mag = np.linalg.norm(gyro, axis=1)
        features.extend(self._extract_channel_features(gyro_mag))

        # Signal Magnitude Area (SMA) - key VISTA feature
        features.append(self._signal_magnitude_area(acc))
        features.append(self._signal_magnitude_area(gyro))

        # Cross-axis correlations (inspired by MSDFE covariance idea)
        for i in range(3):
            for j in range(i+1, 3):
                if len(acc[:, i]) > 1:
                    corr = np.corrcoef(acc[:, i], acc[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        for i in range(3):
            for j in range(i+1, 3):
                if len(gyro[:, i]) > 1:
                    corr = np.corrcoef(gyro[:, i], gyro[:, j])[0, 1]
                    features.append(corr if not np.isnan(corr) else 0.0)
                else:
                    features.append(0.0)

        # Acc-Gyro cross-correlations
        for i in range(3):
            if len(acc[:, i]) > 1:
                corr = np.corrcoef(acc[:, i], gyro[:, i])[0, 1]
                features.append(corr if not np.isnan(corr) else 0.0)
            else:
                features.append(0.0)

        return features

    def extract(self, inertial_data):
        """
        Extract feature vector from a full inertial sequence.

        Strategy: segment into overlapping windows, extract features per window,
        then aggregate window features with statistics.

        Args:
            inertial_data: (samples, C) array — C can be 1 to 6 channels.
                           CZU-MHAD sensor is (10, 1).
        Returns:
            1D feature vector (always same length regardless of input size)
        """
        if inertial_data is None or inertial_data.size == 0:
            return None

        inertial_data = np.nan_to_num(inertial_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure 2D
        if inertial_data.ndim == 1:
            inertial_data = inertial_data.reshape(-1, 1)

        # If shape is (1, N) -> transpose to (N, 1)
        if inertial_data.ndim == 2 and inertial_data.shape[0] < inertial_data.shape[1]:
            inertial_data = inertial_data.T

        # Pad to 6 channels (the feature extractor always expects 6)
        if inertial_data.shape[1] < 6:
            pad_width = 6 - inertial_data.shape[1]
            inertial_data = np.hstack([
                inertial_data,
                np.zeros((inertial_data.shape[0], pad_width))
            ])

        # Apply low-pass Butterworth filter (VISTA paper uses 5Hz cutoff)
        try:
            b, a = signal.butter(4, 0.2, btype='low')  # Normalized frequency
            for ch in range(inertial_data.shape[1]):
                if len(inertial_data[:, ch]) > 12:
                    inertial_data[:, ch] = signal.filtfilt(b, a, inertial_data[:, ch])
        except Exception:
            pass  # Skip filtering if it fails

        # Segment into windows
        n_samples = inertial_data.shape[0]
        step = int(self.window_size * (1 - self.overlap))
        step = max(step, 1)

        windows = []
        for start in range(0, n_samples - self.window_size + 1, step):
            windows.append(inertial_data[start:start + self.window_size])

        if not windows:
            # Sequence shorter than window -> use entire sequence as single window
            windows = [inertial_data]

        # Extract features from each window
        window_features = []
        for w in windows:
            wf = self._extract_window_features(w)
            window_features.append(wf)

        window_features = np.array(window_features)
        n_feat_per_window = window_features.shape[1]

        # ALWAYS aggregate with 4 statistics to ensure consistent output length
        # Even if there's only 1 window, we compute mean/std/min/max (std=0 for 1 window)
        aggregated = []
        for col in range(n_feat_per_window):
            col_data = window_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        return np.nan_to_num(np.array(aggregated), nan=0.0, posinf=0.0, neginf=0.0)


In [7]:
# =============================================================================
# SECTION 5: DEPTH FEATURE EXTRACTION
# =============================================================================

class DepthFeatureExtractor:
    """
    Extracts features from depth map sequences.

    Implements DMM-HOG from CZU-MHAD paper (Chao et al., 2022):
    - Depth Motion Maps (DMM): Project depth frame differences onto
      three orthogonal Cartesian planes (front, side, top)
    - HOG features extracted from each DMM projection
    
    Also retains original features (silhouette, edge, depth stats, temporal)
    for a combined representation.

    References:
    - Chao et al. (2022): CZU-MHAD dataset paper, Section IV-C
    - Chen et al. (2016): DMM method [42] in the paper
    - Yang et al. (2012): DMM-HOG [43] in the paper
    - Siddiqi & Alrashdi (2022): Edge detection features from depth maps
    """

    def __init__(self, config):
        self.n_sample_frames = config.DEPTH_SAMPLE_FRAMES
        # DMM-HOG parameters
        self.dmm_resize = (64, 64)  # Resize DMM maps for consistent HOG output
        self.hog_orientations = 9
        self.hog_pixels_per_cell = (8, 8)
        self.hog_cells_per_block = (2, 2)

    # -------------------------------------------------------------------------
    # DMM-HOG methods (from CZU-MHAD paper)
    # -------------------------------------------------------------------------

    def _compute_dmm(self, depth_data):
        """
        Compute Depth Motion Maps by projecting frame differences onto
        three orthogonal Cartesian planes (front, side, top).

        Given depth volume of shape (frames, H, W):
        - Front projection (xy): collapse along depth axis
        - Side projection (yz): collapse along x axis
        - Top projection (xz): collapse along y axis

        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            dmm_front, dmm_side, dmm_top: three 2D motion maps
        """
        n_frames, H, W = depth_data.shape

        dmm_front = np.zeros((H, W), dtype=np.float64)
        dmm_side = np.zeros((H, 256), dtype=np.float64)  # H x depth_bins
        dmm_top = np.zeros((256, W), dtype=np.float64)    # depth_bins x W

        depth_bins = 256

        for i in range(1, n_frames):
            prev_frame = depth_data[i - 1].astype(np.float64)
            curr_frame = depth_data[i].astype(np.float64)
            diff = np.abs(curr_frame - prev_frame)

            # Front projection (x-y plane): accumulate absolute differences
            dmm_front += diff

            # For side and top projections, we need to bin depth values
            # Side projection (y-z plane): for each row y, accumulate motion
            # across depth bins
            for y in range(H):
                for x in range(W):
                    if diff[y, x] > 0:
                        d_curr = int(curr_frame[y, x])
                        if 0 < d_curr < depth_bins:
                            dmm_side[y, d_curr] += diff[y, x]

            # Top projection (x-z plane): for each col x, accumulate motion
            # across depth bins
            for y in range(H):
                for x in range(W):
                    if diff[y, x] > 0:
                        d_curr = int(curr_frame[y, x])
                        if 0 < d_curr < depth_bins:
                            dmm_top[d_curr, x] += diff[y, x]

        return dmm_front, dmm_side, dmm_top

    def _compute_dmm_fast(self, depth_data):
        """
        Fast vectorized computation of Depth Motion Maps.
        Projects frame differences onto three orthogonal planes.

        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            dmm_front, dmm_side, dmm_top: three 2D motion maps
        """
        n_frames, H, W = depth_data.shape
        depth_bins = 256

        # Compute all frame differences at once
        diffs = np.abs(
            depth_data[1:].astype(np.float64) - depth_data[:-1].astype(np.float64)
        )

        # Front projection (x-y plane): sum of absolute differences per pixel
        dmm_front = np.sum(diffs, axis=0)

        # For side and top projections, use the current frame depth as bin index
        curr_frames = depth_data[1:].astype(np.int32)

        dmm_side = np.zeros((H, depth_bins), dtype=np.float64)
        dmm_top = np.zeros((depth_bins, W), dtype=np.float64)

        # Vectorized side projection: for each (frame, y, x) accumulate
        # diff[frame, y, x] into dmm_side[y, depth_bin]
        for f_idx in range(n_frames - 1):
            diff_frame = diffs[f_idx]
            depth_frame = curr_frames[f_idx]
            mask = (diff_frame > 0) & (depth_frame > 0) & (depth_frame < depth_bins)

            if not np.any(mask):
                continue

            ys, xs = np.where(mask)
            d_vals = depth_frame[ys, xs]
            diff_vals = diff_frame[ys, xs]

            # Side projection: accumulate into (y, depth)
            np.add.at(dmm_side, (ys, d_vals), diff_vals)
            # Top projection: accumulate into (depth, x)
            np.add.at(dmm_top, (d_vals, xs), diff_vals)

        return dmm_front, dmm_side, dmm_top

    def _resize_map(self, img, target_size):
        """
        Resize a 2D map to target_size using simple block averaging/interpolation.
        Uses numpy-only approach to avoid cv2 dependency.
        """
        h, w = img.shape
        th, tw = target_size
        # Simple nearest-neighbor resize
        row_indices = (np.arange(th) * h / th).astype(int)
        col_indices = (np.arange(tw) * w / tw).astype(int)
        row_indices = np.clip(row_indices, 0, h - 1)
        col_indices = np.clip(col_indices, 0, w - 1)
        return img[np.ix_(row_indices, col_indices)]

    def _normalize_map(self, img):
        """Normalize map to [0, 1] range."""
        min_val = np.min(img)
        max_val = np.max(img)
        if max_val - min_val > 1e-12:
            return (img - min_val) / (max_val - min_val)
        return np.zeros_like(img)

    def _compute_hog(self, img):
        """
        Compute HOG (Histogram of Oriented Gradients) features from a 2D image.
        
        Pure numpy implementation following Dalal & Triggs (2005):
        - Compute gradients using [-1, 0, 1] filters
        - Bin gradient orientations into histogram cells
        - Normalize over overlapping blocks
        
        Args:
            img: 2D numpy array (should be resized to self.dmm_resize)
        Returns:
            1D feature vector of HOG descriptors
        """
        img = self._resize_map(img, self.dmm_resize)
        img = self._normalize_map(img)

        H, W = img.shape
        n_orientations = self.hog_orientations
        pix_per_cell_y, pix_per_cell_x = self.hog_pixels_per_cell
        cells_per_block_y, cells_per_block_x = self.hog_cells_per_block

        # Compute gradients
        gx = np.zeros_like(img)
        gy = np.zeros_like(img)
        gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
        gy[1:-1, :] = img[2:, :] - img[:-2, :]

        magnitude = np.sqrt(gx ** 2 + gy ** 2)
        orientation = np.arctan2(gy, gx + 1e-12)
        # Map orientation from [-pi, pi] to [0, pi] (unsigned gradients)
        orientation = orientation % np.pi

        # Compute cell histograms
        n_cells_y = H // pix_per_cell_y
        n_cells_x = W // pix_per_cell_x

        cell_hists = np.zeros((n_cells_y, n_cells_x, n_orientations))

        bin_width = np.pi / n_orientations

        for cy in range(n_cells_y):
            for cx in range(n_cells_x):
                y_start = cy * pix_per_cell_y
                y_end = y_start + pix_per_cell_y
                x_start = cx * pix_per_cell_x
                x_end = x_start + pix_per_cell_x

                cell_mag = magnitude[y_start:y_end, x_start:x_end].ravel()
                cell_ori = orientation[y_start:y_end, x_start:x_end].ravel()

                # Bin orientations with magnitude weighting
                bin_indices = (cell_ori / bin_width).astype(int)
                bin_indices = np.clip(bin_indices, 0, n_orientations - 1)

                for b in range(n_orientations):
                    cell_hists[cy, cx, b] = np.sum(cell_mag[bin_indices == b])

        # Block normalization (L2-norm)
        n_blocks_y = n_cells_y - cells_per_block_y + 1
        n_blocks_x = n_cells_x - cells_per_block_x + 1

        if n_blocks_y <= 0 or n_blocks_x <= 0:
            # Image too small for block normalization, return flattened cell hists
            return cell_hists.ravel()

        hog_features = []
        for by in range(n_blocks_y):
            for bx in range(n_blocks_x):
                block = cell_hists[
                    by : by + cells_per_block_y,
                    bx : bx + cells_per_block_x,
                    :
                ].ravel()
                norm = np.sqrt(np.sum(block ** 2) + 1e-12)
                block = block / norm
                hog_features.extend(block.tolist())

        return np.array(hog_features, dtype=np.float64)

    def _extract_dmm_hog_features(self, depth_data):
        """
        Extract DMM-HOG features from a depth sequence.
        
        Computes Depth Motion Maps on three orthogonal planes,
        then extracts HOG features from each, and concatenates.
        
        Args:
            depth_data: numpy array of shape (frames, H, W)
        Returns:
            1D feature vector (concatenated HOG from 3 DMM projections)
        """
        dmm_front, dmm_side, dmm_top = self._compute_dmm_fast(depth_data)

        hog_front = self._compute_hog(dmm_front)
        hog_side = self._compute_hog(dmm_side)
        hog_top = self._compute_hog(dmm_top)

        return np.concatenate([hog_front, hog_side, hog_top])

    # -------------------------------------------------------------------------
    # Original feature methods (retained for combined representation)
    # -------------------------------------------------------------------------

    def _get_silhouette(self, depth_frame):
        """Extract binary silhouette from depth frame."""
        valid = depth_frame[depth_frame > 0]
        if len(valid) == 0:
            return np.zeros_like(depth_frame, dtype=bool)
        threshold = np.percentile(valid, 50)
        return (depth_frame > 0) & (depth_frame < threshold)

    def _sobel_features(self, frame):
        """
        Compute Sobel edge features (inspired by Siddiqi paper).
        Edge magnitude and direction histograms.
        """
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

        from scipy.ndimage import convolve
        gx = convolve(frame.astype(np.float64), sobel_x)
        gy = convolve(frame.astype(np.float64), sobel_y)

        magnitude = np.sqrt(gx**2 + gy**2)
        direction = np.arctan2(gy, gx + 1e-12)

        features = []

        features.extend([
            np.mean(magnitude),
            np.std(magnitude),
            np.max(magnitude),
            np.sum(magnitude > np.mean(magnitude)),
        ])

        dir_hist, _ = np.histogram(direction[magnitude > np.mean(magnitude)],
                                    bins=8, range=(-np.pi, np.pi))
        if np.sum(dir_hist) > 0:
            dir_hist = dir_hist / (np.sum(dir_hist) + 1e-12)
        features.extend(dir_hist.tolist())

        mag_hist, _ = np.histogram(magnitude.ravel(), bins=8)
        if np.sum(mag_hist) > 0:
            mag_hist = mag_hist / (np.sum(mag_hist) + 1e-12)
        features.extend(mag_hist.tolist())

        return features

    def _shape_features(self, silhouette):
        """Compute shape descriptors from binary silhouette."""
        features = []

        area = np.sum(silhouette)
        features.append(area)

        if area == 0:
            return features + [0] * 7

        rows = np.any(silhouette, axis=1)
        cols = np.any(silhouette, axis=0)
        if np.any(rows) and np.any(cols):
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            height = rmax - rmin + 1
            width = cmax - cmin + 1
            features.append(height)
            features.append(width)
            features.append(height / (width + 1e-6))
            features.append(area / (height * width + 1e-6))
        else:
            features.extend([0, 0, 0, 0])

        y_coords, x_coords = np.where(silhouette)
        if len(y_coords) > 0:
            features.append(np.mean(y_coords) / silhouette.shape[0])
            features.append(np.mean(x_coords) / silhouette.shape[1])
            features.append(np.std(y_coords) / (silhouette.shape[0] + 1e-6))
        else:
            features.extend([0, 0, 0])

        return features

    def _depth_distribution_features(self, depth_frame):
        """Statistical features from depth value distribution."""
        valid = depth_frame[depth_frame > 0].ravel()
        if len(valid) == 0:
            return [0] * 8

        features = [
            np.mean(valid),
            np.std(valid),
            np.median(valid),
            np.min(valid),
            np.max(valid),
            stats.skew(valid) if len(valid) > 2 else 0,
            stats.kurtosis(valid) if len(valid) > 3 else 0,
            np.percentile(valid, 75) - np.percentile(valid, 25),
        ]
        return features

    def _extract_frame_features(self, frame):
        """Extract features from a single depth frame."""
        features = []

        h, w = frame.shape
        scale = 4
        small = frame[::scale, ::scale]

        sil = self._get_silhouette(small)
        features.extend(self._shape_features(sil))

        features.extend(self._sobel_features(small))

        features.extend(self._depth_distribution_features(small))

        return features

    def _load_depth_from_path(self, filepath):
        """Load depth data from .mat file path (lazy loading to save memory)."""
        try:
            mat = loadmat(filepath)
            key = [k for k in mat.keys() if not k.startswith("__")][0]
            return extract_numeric_array(mat[key])
        except Exception as e:
            logger.debug(f"Error loading depth {filepath}: {e}")
            return None

    def extract(self, depth_data):
        """
        Extract feature vector from a depth sequence.

        Combines:
        1. DMM-HOG features (from CZU-MHAD paper) - captures global
           spatio-temporal motion patterns via depth motion maps projected
           onto three orthogonal planes with HOG descriptors
        2. Original per-frame features (silhouette, edge, depth stats,
           temporal) - captures frame-level shape and appearance cues

        Args:
            depth_data: filepath string (lazy load) OR (frames, H, W) array
        Returns:
            1D feature vector: [dmm_hog_features | original_aggregated_features]
            Always same length for a given config.
        """
        # Support lazy loading from file path
        if isinstance(depth_data, str):
            depth_data = self._load_depth_from_path(depth_data)

        if depth_data is None or depth_data.size == 0:
            return None

        depth_data = np.nan_to_num(depth_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ensure shape is (frames, height, width)
        if depth_data.ndim == 3:
            if depth_data.shape[0] > depth_data.shape[2]:
                depth_data = np.transpose(depth_data, (2, 0, 1))
        else:
            return None

        n_frames = depth_data.shape[0]

        if n_frames == 0:
            return None

        # -----------------------------------------------------------------
        # Part 1: DMM-HOG features (paper method)
        # Uses ALL frames for DMM computation (temporal accumulation)
        # -----------------------------------------------------------------
        dmm_hog_features = self._extract_dmm_hog_features(depth_data)

        # -----------------------------------------------------------------
        # Part 2: Original per-frame features (sampled frames)
        # -----------------------------------------------------------------
        if n_frames >= self.n_sample_frames:
            frame_indices = np.linspace(0, n_frames - 1, self.n_sample_frames, dtype=int)
        else:
            frame_indices = np.array([i % n_frames for i in range(self.n_sample_frames)])

        frame_features = []
        for idx in frame_indices:
            ff = self._extract_frame_features(depth_data[idx])
            frame_features.append(ff)

        frame_features = np.array(frame_features)

        # Temporal features from frame differences
        diffs = np.diff(frame_features, axis=0)
        temporal_feats = []
        for col in range(diffs.shape[1]):
            temporal_feats.extend([
                np.mean(diffs[:, col]),
                np.std(diffs[:, col]),
            ])

        # Aggregate frame features
        aggregated = []
        for col in range(frame_features.shape[1]):
            col_data = frame_features[:, col]
            aggregated.extend([
                np.mean(col_data),
                np.std(col_data),
                np.min(col_data),
                np.max(col_data),
            ])

        original_feats = np.concatenate([aggregated, temporal_feats])

        # -----------------------------------------------------------------
        # Combine DMM-HOG + original features
        # -----------------------------------------------------------------
        all_feats = np.concatenate([dmm_hog_features, original_feats])
        del depth_data
        return np.nan_to_num(all_feats, nan=0.0, posinf=0.0, neginf=0.0)

In [8]:
# Video modality not used in CZU-MHAD dataset


In [9]:
# =============================================================================
# SECTION 6: MULTIMODAL FUSION & CLASSIFICATION PIPELINE
# =============================================================================

class MultimodalHARPipeline:
    """
    Complete pipeline: load data, extract features, fuse, classify.
    """

    # Noise strength: std of added noise = NOISE_STRENGTH x std(unit values)
    NOISE_STRENGTH = 1.0

    def __init__(self, config):
        self.config = config
        self.loader = CZUMHADLoader(config)
        self.skel_extractor = CZUSkeletonFeatureExtractor()
        self.iner_extractor = InertialFeatureExtractor(config)
        self.depth_extractor = DepthFeatureExtractor(config)
        self.scaler = StandardScaler()
        # One RNG per modality – seeded from config.LOSS_SEED so results
        # are reproducible, but each advances independently.
        seed = config.LOSS_SEED
        self._skel_noise_rng  = np.random.RandomState(seed)
        self._iner_noise_rng  = np.random.RandomState(seed + 1)
        self._depth_noise_rng = np.random.RandomState(seed + 2)

    def load_all_data(self):
        """Load all modalities via the unified loader."""
        return self.loader.load_all_data()

    def extract_features(self, samples):
        """
        Extract features from all samples across all modalities.
        Applies simulated raw data loss before feature extraction.
        Skips entire sample if ANY modality extraction fails.
        Validates consistent feature dimensions across all samples.

        Returns:
            features_dict: {key: {features, label, subject, modality_features, ...}}
        """
        features_dict = {}
        total = len(samples)
        skipped = 0
        expected_dims = None  # will be set from first successful sample

        for i, (key, sample) in enumerate(samples.items()):
            if (i + 1) % 100 == 0 or i == 0:
                logger.info(f"Extracting features: {i+1}/{total}")

            # -- Skeleton --
            skel = sample['skeleton'].copy()
            apply_skeleton_noise(skel, self.config.LOSS_RATE, self.NOISE_STRENGTH, self._skel_noise_rng)
            skel_feats = self.skel_extractor.extract(skel)
            if skel_feats is None:
                skipped += 1
                continue

            # -- Inertial --
            iner = sample['inertial'].copy()
            apply_inertial_noise(iner, self.config.LOSS_RATE, self.NOISE_STRENGTH, self._iner_noise_rng)
            iner_feats = self.iner_extractor.extract(iner)
            if iner_feats is None:
                skipped += 1
                continue

            # -- Depth --
            depth_feats = None
            if self.config.USE_DEPTH and isinstance(sample['depth'], str):
                depth_raw = self.depth_extractor._load_depth_from_path(sample['depth'])
                if depth_raw is not None:
                    apply_depth_noise(depth_raw, self.config.LOSS_RATE, self.NOISE_STRENGTH, self._depth_noise_rng)
                    depth_feats = self.depth_extractor.extract(depth_raw)
            if depth_feats is None:
                skipped += 1
                continue

            # Check consistent dimensions
            dims = (len(skel_feats), len(iner_feats), len(depth_feats))
            if expected_dims is None:
                expected_dims = dims
                logger.info(f"  Feature dimensions: skeleton={dims[0]}, "
                            f"sensor={dims[1]}, depth={dims[2]}, "
                            f"total={sum(dims)}")
            elif dims != expected_dims:
                logger.warning(f"  Inconsistent dims for {key}: {dims} vs expected {expected_dims}, skipping")
                skipped += 1
                continue

            concatenated = np.concatenate([skel_feats, iner_feats, depth_feats])

            features_dict[key] = {
                'features': concatenated,
                'label': sample['action'],
                'subject': sample['subject'],
                'modalities': ['skeleton', 'inertial', 'depth'],
                'modality_sizes': {
                    'skeleton': len(skel_feats),
                    'inertial': len(iner_feats),
                    'depth': len(depth_feats),
                },
                'modality_features': {
                    'skeleton': skel_feats,
                    'inertial': iner_feats,
                    'depth': depth_feats,
                },
            }

        if skipped > 0:
            logger.warning(f"Skipped {skipped} samples due to extraction failures")
        logger.info(f"Extracted features for {len(features_dict)}/{total} samples")

        if features_dict:
            sample_entry = next(iter(features_dict.values()))
            logger.info(f"Total feature vector dimension: {len(sample_entry['features'])}")
            for mod, size in sample_entry['modality_sizes'].items():
                logger.info(f"  {mod}: {size} features")

        return features_dict

    def save_features(self, features_dict):
        """
        Save extracted features for later use.

        Saves:
          - X_feat.pkl: list of dicts with 'depth_feat', 'sensor_feat', 'skeleton_feat'
          - y.npy: encoded action labels
          - subjects.npy: subject IDs per sample
          - label_encoder.pkl: fitted LabelEncoder
        """
        out_dir = self.config.FEATURES_DIR
        os.makedirs(out_dir, exist_ok=True)

        keys = sorted(features_dict.keys())

        X_feat = []
        y_raw = []
        subjects = []

        for k in keys:
            entry = features_dict[k]
            mod_feats = entry.get('modality_features', {})

            sample_dict = {
                'skeleton_feat': mod_feats.get('skeleton', np.array([])),
                'sensor_feat':   mod_feats.get('inertial', np.array([])),
                'depth_feat':    mod_feats.get('depth', np.array([])),
            }
            X_feat.append(sample_dict)
            y_raw.append(entry['label'])
            subjects.append(entry['subject'])

        y_raw = np.array(y_raw)
        subjects = np.array(subjects)

        le = LabelEncoder()
        y = le.fit_transform(y_raw)

        joblib.dump(X_feat, os.path.join(out_dir, 'X_feat.pkl'))
        np.save(os.path.join(out_dir, 'y.npy'), y)
        np.save(os.path.join(out_dir, 'subjects.npy'), subjects)
        joblib.dump(le, os.path.join(out_dir, 'label_encoder.pkl'))

        logger.info(f"Features saved to '{out_dir}/':")
        logger.info(f"  X_feat.pkl:        {len(X_feat)} samples")
        first = X_feat[0]
        for mod_key in ['skeleton_feat', 'sensor_feat', 'depth_feat']:
            dim = first[mod_key].shape[0] if first[mod_key].size > 0 else 0
            logger.info(f"    {mod_key}: {dim} features")
        total_dim = sum(first[m].shape[0] for m in first if first[m].size > 0)
        logger.info(f"    TOTAL: {total_dim} features")
        logger.info(f"  y.npy:             {y.shape} (encoded, {len(le.classes_)} classes)")
        logger.info(f"  subjects.npy:      {subjects.shape} "
                     f"(subjects: {sorted(np.unique(subjects).tolist())})")
        logger.info(f"  label_encoder.pkl: classes = {le.classes_.tolist()}")

    def split_train_test(self, features_dict):
        """Split into train/test using subject-based protocol."""
        X_train, y_train = [], []
        X_test, y_test = [], []

        for key, entry in features_dict.items():
            if entry['subject'] in self.config.TRAIN_SUBJECTS:
                X_train.append(entry['features'])
                y_train.append(entry['label'])
            elif entry['subject'] in self.config.TEST_SUBJECTS:
                X_test.append(entry['features'])
                y_test.append(entry['label'])

        X_train = np.array(X_train, dtype=np.float64)
        y_train = np.array(y_train)
        X_test = np.array(X_test, dtype=np.float64)
        y_test = np.array(y_test)

        logger.info(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
        logger.info(f"Feature dimension: {X_train.shape[1]}")
        logger.info(f"Number of classes: {len(np.unique(y_train))}")

        return X_train, y_train, X_test, y_test

    def train_and_evaluate(self, X_train, y_train, X_test, y_test):
        """Train Random Forest and evaluate."""
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

        rf = RandomForestClassifier(
            n_estimators=self.config.RF_N_ESTIMATORS,
            max_depth=self.config.RF_MAX_DEPTH,
            min_samples_split=self.config.RF_MIN_SAMPLES_SPLIT,
            min_samples_leaf=self.config.RF_MIN_SAMPLES_LEAF,
            random_state=self.config.RF_RANDOM_STATE,
            n_jobs=self.config.RF_N_JOBS,
            class_weight='balanced',
        )

        logger.info("Training Random Forest classifier...")
        start_time = time.time()
        rf.fit(X_train_scaled, y_train)
        train_time = time.time() - start_time
        logger.info(f"Training completed in {train_time:.2f} seconds")

        y_pred = rf.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro')
        f1_weighted = f1_score(y_test, y_pred, average='weighted')

        logger.info(f"\n{'='*60}")
        logger.info(f"RESULTS")
        logger.info(f"{'='*60}")
        logger.info(f"Overall Accuracy: {accuracy*100:.2f}%")
        logger.info(f"Macro F1-Score:   {f1_macro*100:.2f}%")
        logger.info(f"Weighted F1-Score: {f1_weighted*100:.2f}%")

        action_names = [f"Action {i}" for i in range(1, self.config.NUM_ACTIONS + 1)]
        unique_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
        target_names = [action_names[l-1] for l in unique_labels]

        report = classification_report(y_test, y_pred, labels=unique_labels,
                                        target_names=target_names, digits=3)
        logger.info(f"\nClassification Report:\n{report}")

        logger.info("Running 5-fold cross-validation on training set...")
        cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring='accuracy')
        logger.info(f"CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")

        importances = rf.feature_importances_
        top_k = 20
        top_indices = np.argsort(importances)[-top_k:][::-1]
        logger.info(f"\nTop {top_k} most important features (by index):")
        for idx in top_indices:
            logger.info(f"  Feature {idx}: importance = {importances[idx]:.4f}")

        return {
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'train_time': train_time,
            'model': rf,
            'y_pred': y_pred,
        }

    def run(self):
        """Execute the full pipeline."""
        logger.info("="*60)
        logger.info("MULTIMODAL HAR PIPELINE FOR CZU-MHAD  (Gaussian noise)")
        logger.info("="*60)
        logger.info(f"Modalities: Skeleton={self.config.USE_SKELETON}, "
                     f"Inertial={self.config.USE_INERTIAL}, "
                     f"Depth={self.config.USE_DEPTH}")
        logger.info(f"Noise rate : {self.config.LOSS_RATE*100:.0f}% of structural units  "
                     f"(noise_strength={self.NOISE_STRENGTH})")
        logger.info(f"Train subjects: {self.config.TRAIN_SUBJECTS}")
        logger.info(f"Test subjects:  {self.config.TEST_SUBJECTS}")

        logger.info("\n--- Step 1: Loading data ---")
        samples = self.load_all_data()

        if not samples:
            logger.error("No data loaded! Check the dataset path.")
            logger.info(f"Expected path: {self.config.DATASET_ROOT}")
            return None

        logger.info("\n--- Step 2: Extracting features (with Gaussian noise applied) ---")
        features_dict = self.extract_features(samples)

        if not features_dict:
            logger.error("No features extracted!")
            return None

        logger.info("\n--- Step 2b: Saving features ---")
        self.save_features(features_dict)

        logger.info("\n--- Step 3: Train/test split ---")
        X_train, y_train, X_test, y_test = self.split_train_test(features_dict)

        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            logger.error("Empty train or test set!")
            return None

        logger.info("\n--- Step 4: Training and evaluation ---")
        results = self.train_and_evaluate(X_train, y_train, X_test, y_test)

        return results


In [10]:
# =============================================================================
# SECTION 7: DEMO MODE (generates synthetic data for testing)
# =============================================================================

def create_synthetic_dataset(root_dir, n_actions=22, subjects=None, n_trials=5):
    """Create synthetic CZU-MHAD-like dataset for testing."""
    if subjects is None:
        subjects = ['cx', 'myj', 'zyh', 'cyy', 'qyh']

    logger.info("Creating synthetic dataset for demonstration...")

    for modality, subdir in [("skeleton", "skeleton_mat"), ("sensor", "sensor_mat"),
                              ("depth", "depth_mat")]:
        mod_dir = os.path.join(root_dir, subdir)
        os.makedirs(mod_dir, exist_ok=True)

        for a in range(1, n_actions + 1):
            for s in subjects:
                for t in range(1, n_trials + 1):
                    fname = f"{s}_a{a}_t{t}.mat"
                    fpath = os.path.join(mod_dir, fname)

                    if os.path.exists(fpath):
                        continue

                    from scipy.io import savemat

                    if modality == "skeleton":
                        # (n_frames, 100) synthetic skeleton
                        n_frames = np.random.randint(30, 80)
                        base = np.random.randn(n_frames, 100) * 0.1
                        base[:, 0] += np.sin(np.linspace(0, a * np.pi, n_frames))
                        savemat(fpath, {'skeleton': base})

                    elif modality == "sensor":
                        # (10, 1) sensor data
                        data = np.random.randn(10, 1) * 0.5
                        data[:, 0] += np.sin(np.linspace(0, a * 0.5, 10))
                        savemat(fpath, {'sensor': data})

                    elif modality == "depth":
                        # (n_depth_frames, H, W) depth data
                        h, w = 106, 128
                        n_depth_frames = 20
                        depth = np.zeros((n_depth_frames, h, w))
                        for fr in range(n_depth_frames):
                            cy = h // 2 + int(5 * np.sin(a * fr / 10.0))
                            cx_pos = w // 2 + int(5 * np.cos(a * fr / 10.0))
                            Y, X = np.ogrid[:h, :w]
                            mask = (Y - cy)**2 + (X - cx_pos)**2 < (10 + a)**2
                            depth[fr][mask] = 1000 + a * 50 + np.random.rand() * 100
                        savemat(fpath, {'depth': depth})

    logger.info(f"Synthetic dataset created at {root_dir}")


# =============================================================================
# SECTION 8: MAIN EXECUTION
# =============================================================================

def main():
    """Main entry point."""
    config = Config()

    if not os.path.isdir(config.DATASET_ROOT):
        logger.info(f"Dataset not found at '{config.DATASET_ROOT}'")
        logger.info("Creating synthetic dataset for demonstration...")
        os.makedirs(config.DATASET_ROOT, exist_ok=True)
        create_synthetic_dataset(config.DATASET_ROOT)

    pipeline = MultimodalHARPipeline(config)
    results = pipeline.run()

    if results:
        logger.info(f"\n{'='*60}")
        logger.info("FINAL SUMMARY")
        logger.info(f"{'='*60}")
        logger.info(f"Accuracy:         {results['accuracy']*100:.2f}%")
        logger.info(f"Macro F1:         {results['f1_macro']*100:.2f}%")
        logger.info(f"Weighted F1:      {results['f1_weighted']*100:.2f}%")
        logger.info(f"CV Accuracy:      {results['cv_mean']*100:.2f}% "
                     f"(+/- {results['cv_std']*100:.2f}%)")
        logger.info(f"Training Time:    {results['train_time']:.2f}s")


if __name__ == "__main__":
    main()


2026-05-27 02:23:46,058 - INFO - ============================================================
2026-05-27 02:23:46,059 - INFO - MULTIMODAL HAR PIPELINE FOR CZU-MHAD  (Gaussian noise)
2026-05-27 02:23:46,059 - INFO - ============================================================
2026-05-27 02:23:46,060 - INFO - Modalities: Skeleton=True, Inertial=True, Depth=True
2026-05-27 02:23:46,060 - INFO - Noise rate : 30% of structural units  (noise_strength=1.0)
2026-05-27 02:23:46,061 - INFO - Train subjects: ['cx', 'myj', 'zyh']
2026-05-27 02:23:46,061 - INFO - Test subjects:  ['cyy', 'qyh']
2026-05-27 02:23:46,062 - INFO - 
--- Step 1: Loading data ---
2026-05-27 02:23:46,065 - INFO - Subjects found: ['cx', 'cyy', 'myj', 'qyh', 'zyh']
2026-05-27 02:23:46,066 - INFO - Total .mat files in sensor_mat: 1165
2026-05-27 02:23:46,090 - INFO - Samples with all 3 modalities: 1165
2026-05-27 02:23:46,142 - INFO -   DEBUG depth_mat/cx_a10_t1.mat key='depth': type=ndarray, dtype=uint8, shape=(132, 424, 512)